# Proposal Atypicality RA Handoff Notebook

This notebook is a clean handoff version of the proposal atypicality pipeline. The unit of analysis is the proposal. The core measurement follows Uzzi et al.: identify journals cited by a proposal, form all cited-journal pairs within that proposal, then score those pairs against a global/year-specific background distribution of journal-pair combinations.

The current notebook focuses on the proposal side: parsing references, extracting journal strings, matching them to OpenAlex/SciSciNet source IDs, exporting diagnostics, and building proposal-level source pairs. It does not estimate the background expected values yet.


## Task Overview

Goal: calculate an atypicality/novelty score for each proposal based on the papers it cites in `Project literature references clean`.

Important constraint: only journals resolved to OpenAlex/SciSciNet source IDs are scoreable. Unmatched journal strings are not useful for the final Uzzi-style score, because local IDs will not appear in a global SciSciNet/OpenAlex background z-score table. Unmatched strings are exported for review instead.


## 0. Setup and Paths

All data files should be in the same directory as this notebook. The path names below are in a local or cluster folder without editing absolute paths.


In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import time
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from itertools import combinations

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display

# All input files are assumed to live in the same directory as this notebook.
# Move/copy the files listed below into this directory, or adjust only WORK_DIR.
WORK_DIR = Path.cwd()
INPUT_PARQUET = WORK_DIR / 'research_project_texts_2006-2026.parquet'

# Authority/source files. OpenAlex/SciSciNet source IDs are the IDs needed for final scoring.
OPENALEX_SOURCES_PATH = WORK_DIR / 'openalex_sources_journals.parquet'  # optional
SCISCINET_SOURCES_PATH = WORK_DIR / 'sciscinet_sources.parquet'          # recommended
NNF_JOURNALS_PATH = WORK_DIR / 'List of Journals of Papers Produced by NNF Proposals - 2026-06-16.parquet'  # helper aliases
WOS_JCR_PATH = WORK_DIR / 'wos_jcr.csv'                                  # helper aliases
MANUAL_JOURNAL_MATCH_CSV_PATH = WORK_DIR / 'manual_journal_matches.csv'  # optional RA corrections

# Future background/null-model inputs, not used until the z-score table is built.
SCISCINET_PAPERREFS_PATH = WORK_DIR / 'sciscinet_paperrefs.parquet'
SCISCINET_PAPERS_PATH = WORK_DIR / 'sciscinet_papers.parquet'
SCISCINET_PAPERSOURCES_PATH = WORK_DIR / 'sciscinet_papersources.parquet'

ID_COL = 'Application id'
YEAR_COL = 'app year'
REF_COL = 'Project literature references clean'

INTERMEDIATE_DIR = WORK_DIR / 'intermediate' / 'proposal_atypicality_ra_handoff'
OUTPUT_DIR = WORK_DIR / 'output' / 'proposal_atypicality_ra_handoff'
CSV_DIR = INTERMEDIATE_DIR / 'csv'
for path in [INTERMEDIATE_DIR, OUTPUT_DIR, CSV_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_REFS_PATH = INTERMEDIATE_DIR / 'proposal_references_raw.parquet'
PARSED_REFS_PATH = INTERMEDIATE_DIR / 'proposal_references_parsed.parquet'
PARSING_FAILURES_PATH = INTERMEDIATE_DIR / 'proposal_references_parsing_failures.parquet'
PARSER_STATUS_PATH = OUTPUT_DIR / 'reference_parser_status.csv'
PARSER_USAGE_PATH = OUTPUT_DIR / 'reference_parser_usage_summary.csv'
JOURNAL_AUTHORITY_PATH = INTERMEDIATE_DIR / 'journal_authority.parquet'
JOURNAL_AUTHORITY_ALIASES_PATH = INTERMEDIATE_DIR / 'journal_authority_aliases.parquet'
REFERENCE_JOURNAL_MATCHES_PATH = INTERMEDIATE_DIR / 'proposal_reference_journal_matches.parquet'
UNMATCHED_REFERENCE_JOURNAL_STRINGS_PATH = INTERMEDIATE_DIR / 'unmatched_reference_journal_strings.parquet'
PROPOSAL_MATCH_SUMMARY_PATH = OUTPUT_DIR / 'proposal_reference_journal_match_summary.parquet'
PROPOSAL_JOURNAL_PAIRS_PATH = INTERMEDIATE_DIR / 'proposal_openalex_source_pairs.parquet'
SCORED_PAIRS_PATH = INTERMEDIATE_DIR / 'proposal_openalex_source_pairs_scored.parquet'
PROPOSAL_SCORES_PATH = OUTPUT_DIR / 'proposal_atypicality_scores.parquet'
DEDUP_MATCHED_JOURNALS_PATH = OUTPUT_DIR / 'deduplicated_reference_journal_matches.parquet'
DEDUP_UNMATCHED_JOURNALS_PATH = OUTPUT_DIR / 'deduplicated_unmatched_reference_journals.parquet'

print('WORK_DIR:', WORK_DIR)
print('INPUT_PARQUET exists:', INPUT_PARQUET.exists(), INPUT_PARQUET)
print('OpenAlex source file exists:', OPENALEX_SOURCES_PATH.exists(), OPENALEX_SOURCES_PATH)
print('SciSciNet source file exists:', SCISCINET_SOURCES_PATH.exists(), SCISCINET_SOURCES_PATH)
print('SciSciNet paper-source file exists:', SCISCINET_PAPERSOURCES_PATH.exists(), SCISCINET_PAPERSOURCES_PATH)


## 1. Parser Activation Signals

AnyStyle can parse semi-structured references. If a required parser is inactive, the notebook prints a status table and stops before producing misleading downstream matches.


In [ ]:
USE_ANYSTYLE = True
REQUIRE_ANYSTYLE = True
ANYSTYLE_BIN = str(WORK_DIR / 'envs' / 'ruby_anystyle_32' / 'share' / 'rubygems' / 'bin' / 'anystyle')
if not Path(ANYSTYLE_BIN).exists():
    ANYSTYLE_BIN = 'anystyle'

ALLOW_REGEX_FALLBACK = True
STOP_IF_REQUIRED_PARSER_INACTIVE = True

parser_status = {
    'anystyle': {'requested': USE_ANYSTYLE, 'required': REQUIRE_ANYSTYLE, 'active': False, 'message': ''},
}

if USE_ANYSTYLE:
    anystyle_path = shutil.which(ANYSTYLE_BIN)
    parser_status['anystyle']['active'] = anystyle_path is not None
    parser_status['anystyle']['message'] = anystyle_path or 'not found on PATH'
else:
    parser_status['anystyle']['message'] = 'not requested'

parser_status_df = pd.DataFrame(parser_status).T
parser_status_df.to_csv(PARSER_STATUS_PATH)
display(parser_status_df)

inactive_required = parser_status_df[
    parser_status_df['requested'].astype(bool)
    & parser_status_df['required'].astype(bool)
    & ~parser_status_df['active'].astype(bool)
]

if len(inactive_required):
    print('STOP: Required parser(s) are inactive. Troubleshoot before continuing.')
    display(inactive_required)
    if STOP_IF_REQUIRED_PARSER_INACTIVE:
        raise SystemExit('Required parser inactive')
else:
    print('Parser activation check passed.')


## 2. Load Proposal Data and Split Reference Lists

This reads proposal rows and splits `Project literature references clean` into candidate references. The split is heuristic because the references are semi-structured and vary by proposal.


In [ ]:
df = pd.read_parquet(INPUT_PARQUET, engine='fastparquet')
refs = df[[ID_COL, YEAR_COL, REF_COL]].copy()
refs[REF_COL] = refs[REF_COL].fillna('').astype(str)
refs_nonempty = refs[refs[REF_COL].str.strip().ne('')].copy()

print('proposal rows loaded:', len(df))
print('unique application IDs:', df[ID_COL].nunique(dropna=True))
print('non-empty reference-list rows:', len(refs_nonempty))
print('reference length summary:')
print(refs[REF_COL].str.len().describe())

DOI_RE = re.compile(r'10\.\d{4,9}/[-._;()/:A-Z0-9]+', re.I)
PMID_RE = re.compile(r'\bPMID\s*:?\s*(\d+)\b', re.I)
PMCID_RE = re.compile(r'\b(PMC\d+)\b', re.I)
YEAR_RE = re.compile(r'(?<!\d)(19\d{2}|20\d{2})(?!\d)')

def normalize_space(text):
    text = str(text).replace('\r', '\n').replace('\u00a0', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def normalize_doi(doi):
    doi = str(doi).strip()
    doi = re.sub(r'^https?://(dx\.)?doi\.org/', '', doi, flags=re.I)
    doi = re.sub(r'^doi\s*:\s*', '', doi, flags=re.I)
    return doi.strip().rstrip('.,;:)\]}>').lower()

def split_references(text):
    text = normalize_space(text)
    if not text:
        return []
    text = re.sub(r'^(references|bibliography)\s*[:\n]+', '', text, flags=re.I).strip()
    numbered = re.split(r'(?m)(?=^\s*(?:\[\d{1,3}\]|\(\d{1,3}\)|\*?\d{1,3}[\.)])\s+)', text)
    numbered = [x.strip() for x in numbered if len(x.strip()) > 20]
    if len(numbered) >= 2:
        return numbered
    blankline = re.split(r'\n\s*\n+', text)
    blankline = [x.strip() for x in blankline if len(x.strip()) > 20]
    if len(blankline) >= 2:
        return blankline
    newline = re.split(r'\n+', text)
    newline = [x.strip() for x in newline if len(x.strip()) > 40]
    if len(newline) >= 2:
        return newline
    return [text]

raw_rows = []
for _, row in tqdm(refs_nonempty.iterrows(), total=len(refs_nonempty), desc='Splitting proposal references'):
    for pos, entry in enumerate(split_references(row[REF_COL]), start=1):
        raw_rows.append({ID_COL: row[ID_COL], YEAR_COL: row[YEAR_COL], 'reference_position': pos, 'reference_text': entry})

raw_refs = pd.DataFrame(raw_rows)
raw_refs.to_parquet(RAW_REFS_PATH, index=False)
print('raw parsed reference entries:', len(raw_refs))
raw_refs.head()


## 3. Parse References

AnyStyle is called in **batch mode** (5000 refs per subprocess call) for speed — parsing 600k references takes ~12 minutes this way vs. hours per-reference. Results are cached to `PARSED_REFS_PATH`; delete that file to re-run parsing. The most important output column is `parsed_journal_raw`, the journal string matched in Step 5.

In [ ]:
import tempfile

MONTH_RE = re.compile(r'^(jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december)$', re.I)

def has_value(x):
    return pd.notna(x) and str(x).strip() != ''

def classify_reference(ref_text, parsed_journal_raw=None, parsed_doi=None):
    ref_text = '' if pd.isna(ref_text) else str(ref_text)
    labels = []
    lower = ref_text.lower()
    if re.search(r'\b(book|chapter|edited by|publisher|press|isbn)\b', lower): labels.append('book_signal')
    if re.search(r'\b(thesis|dissertation)\b', lower): labels.append('thesis_signal')
    if re.search(r'\b(report|working paper|white paper|technical report)\b', lower): labels.append('report_signal')
    if re.search(r'\b(conference|proceedings|symposium|workshop)\b', lower): labels.append('conference_signal')
    has_paper_signal = bool(re.search(r'\b(journal|vol\.|volume|issue|pages?|pp\.|doi)\b', lower))
    has_pages_or_volume = bool(re.search(r'\b\d{1,4}\s*\(?\d{0,4}\)?\s*[:,]\s*\d{1,6}', ref_text))
    has_year = bool(YEAR_RE.search(ref_text))
    has_doi = has_value(parsed_doi) or bool(DOI_RE.search(ref_text))
    has_journal = has_value(parsed_journal_raw)
    if has_journal or has_doi or has_paper_signal or has_pages_or_volume:
        ref_type = 'likely_paper'
    elif labels:
        ref_type = 'likely_nonpaper'
    elif has_year:
        ref_type = 'ambiguous'
    else:
        ref_type = 'unclassified'
    return ref_type, ';'.join(labels)

def anystyle_item_to_parsed(item):
    journal = item.get('container-title') or item.get('journal') or item.get('collection-title')
    title = item.get('title')
    if isinstance(title, list): title = ' '.join(map(str, title))
    if isinstance(journal, list): journal = ' '.join(map(str, journal))
    return {
        'parsed_title': title if title else pd.NA,
        'parsed_journal_raw': journal if journal else pd.NA,
        'parsed_year': item.get('date') or item.get('year') or pd.NA,
        'parsed_doi': normalize_doi(item.get('doi')) if item.get('doi') else pd.NA,
        'parsed_volume': item.get('volume') or pd.NA,
        'parsed_issue': item.get('issue') or pd.NA,
        'parsed_pages': item.get('pages') or item.get('page') or pd.NA,
    }

def batch_anystyle(reference_texts, tmp_dir, batch_size=5000):
    """Parse references in batched AnyStyle subprocess calls (one call per batch).
    
    AnyStyle requires one reference per line; internal newlines are stripped before writing.
    """
    all_results = []
    for i in range(0, len(reference_texts), batch_size):
        batch = reference_texts[i:i+batch_size]
        cleaned = [re.sub(r'\s+', ' ', t).strip() for t in batch]
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', dir=tmp_dir, delete=False, encoding='utf-8') as f:
            f.write('\n'.join(cleaned))
            tmp_in = f.name
        try:
            proc = subprocess.run(
                [ANYSTYLE_BIN, '--stdout', '-f', 'json', 'parse', tmp_in],
                capture_output=True, text=True, timeout=600
            )
            if proc.returncode != 0 or not proc.stdout.strip():
                all_results.extend([{}] * len(batch))
                continue
            results = json.loads(proc.stdout)
            if not isinstance(results, list):
                all_results.extend([{}] * len(batch))
                continue
            while len(results) < len(batch):
                results.append({})
            all_results.extend(results[:len(batch)])
        except Exception:
            all_results.extend([{}] * len(batch))
        finally:
            try: os.unlink(tmp_in)
            except: pass
    return all_results

def clean_candidate_span(span):
    span = str(span)
    span = re.sub(r'^\s*(in\s+)?', '', span, flags=re.I)
    span = re.sub(r'\b(doi|pmid|pmcid)\b.*$', '', span, flags=re.I)
    span = re.sub(r'\b(?:19\d{2}|20\d{2})\b.*$', '', span)
    span = re.sub(r'\b\d{1,4}\s*\(?\d{0,4}\)?\s*[:,].*$', '', span)
    span = re.sub(r'[,.;:()\[\]"""]+$', '', span).strip()
    span = re.sub(r'^[,.;:()\[\]"""]+', '', span).strip()
    return re.sub(r'\s+', ' ', span)

def regex_journal_candidates(ref_text):
    text = normalize_space(ref_text).replace('\n', ' ')
    text = re.sub(r'^\s*(?:\[\d+\]|\(\d+\)|\*?\d+[\.)])\s*', '', text)
    candidates = []
    patterns = [
        r'\.\s*([^\.]{2,90}?)\.\s*(?:19\d{2}|20\d{2})\b',
        r'\.\s*([^\.]{2,90}?),\s*(?:19\d{2}|20\d{2})\b',
        r'\.\s*([^\.]{2,70}?)\s+\d{1,4}\s*[,;:]\s*\d+',
        r'\b([A-Z][A-Za-z&\. ]{2,70}?)\s+\d{1,4}\s*,\s*\d+',
        r'\.\s*([^\.]{2,90}?),\s*\d{1,4}\s*\(',
        r'\.\s*([^\.]{2,90}?),\s*\d{1,4}\s*[:,]',
    ]
    for pat in patterns:
        for match in re.finditer(pat, text):
            cand = clean_candidate_span(match.group(1))
            if 2 <= len(cand) <= 100:
                candidates.append(cand)
    cleaned = []
    bad = re.compile(r'\b(et al|abstract|appendix|table|figure|references|retrieved|available|http|doi)\b', re.I)
    for cand in candidates:
        toks = cand.split()
        if bad.search(cand) or MONTH_RE.match(cand):
            continue
        if len(toks) <= 12 and any(re.search('[A-Za-z]', t) for t in toks):
            cleaned.append(cand)
    return list(dict.fromkeys(cleaned))

if PARSED_REFS_PATH.exists():
    print(f'Loading cached parsed refs from {PARSED_REFS_PATH}')
    parsed_refs = pd.read_parquet(PARSED_REFS_PATH)
else:
    tmp_dir = tempfile.mkdtemp()
    texts = raw_refs['reference_text'].tolist()
    all_anystyle = []

    if USE_ANYSTYLE and bool(parser_status_df.loc['anystyle', 'active']):
        print(f'Running AnyStyle in batches of 5000 on {len(texts):,} references (~12 min)...')
        for i in tqdm(range(0, len(texts), 5000), desc='AnyStyle batches'):
            batch = texts[i:i+5000]
            all_anystyle.extend(batch_anystyle(batch, tmp_dir, batch_size=len(batch)))
        print(f'AnyStyle done. Got {len(all_anystyle)} results.')
    else:
        print('AnyStyle inactive, using regex fallback only.')
        all_anystyle = [{}] * len(texts)

    parsed_rows = []
    for idx, row in enumerate(tqdm(raw_refs.itertuples(index=False), total=len(raw_refs), desc='Building parsed rows')):
        item = all_anystyle[idx] if idx < len(all_anystyle) else {}
        if item:
            parsed = anystyle_item_to_parsed(item)
            parsed['parser_used'] = 'anystyle' if pd.notna(parsed.get('parsed_journal_raw')) else 'anystyle_no_journal'
            parsed['parse_success'] = pd.notna(parsed.get('parsed_journal_raw'))
            parsed['parse_error'] = ''
        else:
            parsed = {'parsed_title': pd.NA, 'parsed_journal_raw': pd.NA, 'parsed_year': pd.NA,
                      'parsed_doi': pd.NA, 'parsed_volume': pd.NA, 'parsed_issue': pd.NA,
                      'parsed_pages': pd.NA, 'parser_used': 'none', 'parse_success': False, 'parse_error': ''}

        if not pd.notna(parsed.get('parsed_journal_raw')) and ALLOW_REGEX_FALLBACK:
            candidates = regex_journal_candidates(row.reference_text)
            if candidates:
                parsed['parsed_journal_raw'] = candidates[0]
                parsed['parser_used'] = 'regex_fallback'
                parsed['parse_success'] = True

        dois  = sorted(set(normalize_doi(x) for x in DOI_RE.findall(row.reference_text)))
        dois  = [x for x in dois if x.startswith('10.')]
        pmids  = sorted(set(PMID_RE.findall(row.reference_text)))
        pmcids = sorted(set(x.upper() for x in PMCID_RE.findall(row.reference_text)))
        years  = YEAR_RE.findall(row.reference_text)
        ref_type, labels = classify_reference(row.reference_text, parsed.get('parsed_journal_raw'), parsed.get('parsed_doi'))
        parsed_rows.append({
            ID_COL: getattr(row, '_0'), YEAR_COL: getattr(row, '_1'),
            'reference_position': row.reference_position,
            'reference_text': row.reference_text,
            'dois': dois, 'pmids': pmids, 'pmcids': pmcids,
            'reference_year_guess': int(years[-1]) if years else pd.NA,
            'reference_type': ref_type, 'nonpaper_labels': labels,
            'is_paper_candidate': ref_type in ['likely_paper', 'unknown_with_year'],
            'has_doi': bool(dois), **parsed,
        })

    parsed_refs = pd.DataFrame(parsed_rows)
    for col in ['parsed_volume', 'parsed_issue', 'parsed_pages', 'parsed_year']:
        if col in parsed_refs.columns:
            parsed_refs[col] = parsed_refs[col].astype(str).where(parsed_refs[col].notna(), pd.NA)
    parsed_refs.to_parquet(PARSED_REFS_PATH, index=False)
    failures = parsed_refs[~parsed_refs['parse_success'] | parsed_refs['parsed_journal_raw'].isna()].copy()
    failures.to_parquet(PARSING_FAILURES_PATH, index=False)
    parser_usage = parsed_refs.groupby('parser_used', dropna=False).agg(
        references=('reference_text', 'size'),
        parse_success=('parse_success', 'sum'),
        parsed_journals=('parsed_journal_raw', lambda x: x.notna().sum()),
    ).reset_index()
    parser_usage.to_csv(PARSER_USAGE_PATH, index=False)
    display(parser_usage)

print('parsed reference entries:', len(parsed_refs))
print('likely paper references:', int(parsed_refs['is_paper_candidate'].sum()))
print('references with parsed journal:', int(parsed_refs['parsed_journal_raw'].notna().sum()))


## 4. Build Journal Authority and Aliases

**Authority source priority:** If `sciscinet_sources.parquet` is available (Moh's cluster), it is used first and provides real OpenAlex source IDs (e.g. `S2764573992`). If only `openalex_sources_journals.parquet` is present (built from Crossref via `fetch_journals_crossref.py`), the `openalex_source_id` column contains **ISSN-L** values rather than OpenAlex source IDs — this is a proxy used because OpenAlex's API was rate-limited during initial development.

**⚠️ ISSN-L vs. OpenAlex source ID:** If the authority was built from Crossref (the fallback path), `journal_1`/`journal_2` in the output pairs file are ISSN-L strings (e.g. `0028-0836`), not OpenAlex IDs. Before joining to the SciSciNet background table, map ISSN-L → OpenAlex source ID using `sciscinet_papersources.parquet` or the SciSciNet sources file.

**Building the authority on a new machine:** Run `fetch_journals_crossref.py` in the `Notebook/` directory. It downloads ~167k journals from Crossref (no API key needed, ~10 min) and writes `Data/openalex_sources_journals.parquet`. For best results, use `sciscinet_sources.parquet` instead — it has real OpenAlex source IDs and no remapping is needed.

In [ ]:
ABBREV = {
    'journal': 'j', 'journals': 'j', 'proceedings': 'proc', 'proceeding': 'proc',
    'national': 'natl', 'academy': 'acad', 'academies': 'acad',
    'sciences': 'sci', 'science': 'sci', 'nature': 'nat', 'nat.': 'nat', 'springer nature': 'nat',
    'scientific': 'sci', 'united': 'u', 'states': 's', 'america': 'a', 'american': 'am',
    'british': 'br', 'european': 'eur', 'international': 'int', 'clinical': 'clin',
    'clinic': 'clin', 'medicine': 'med', 'medical': 'med', 'biology': 'biol', 'biological': 'biol',
    'biochemistry': 'biochem', 'biochemical': 'biochem', 'chemistry': 'chem', 'chemical': 'chem',
    'molecular': 'mol', 'cellular': 'cell', 'genetics': 'genet', 'genetic': 'genet',
    'immunology': 'immunol', 'microbiology': 'microbiol', 'physiology': 'physiol',
    'pharmacology': 'pharmacol', 'endocrinology': 'endocrinol', 'metabolism': 'metab',
    'neuroscience': 'neurosci', 'neurology': 'neurol', 'cardiology': 'cardiol',
    'cardiovascular': 'cardiovasc', 'oncology': 'oncol', 'hematology': 'hematol',
    'haematology': 'haematol', 'physics': 'phys', 'physical': 'phys', 'review': 'rev',
    'reviews': 'rev', 'letters': 'lett', 'materials': 'mater', 'applied': 'appl',
    'environmental': 'environ', 'technology': 'technol', 'biotechnology': 'biotechnol',
    'research': 'res', 'reports': 'rep', 'communications': 'commun', 'current': 'curr',
    'opinion': 'opin', 'development': 'dev', 'experimental': 'exp', 'therapy': 'ther',
    'translational': 'transl', 'epidemiology': 'epidemiol', 'nutrition': 'nutr', 'respiratory': 'respir',
    'computational': 'comput', 'systems': 'syst', 'applications': 'appl', 'engineering': 'eng', 'engineer': 'eng',
}

STOPWORDS = {'of', 'the', 'and', 'for', 'in', 'on', 'to', 'a', 'an', '&'}

# Hand-curated aliases for common abbreviated or non-standard journal strings found in
# the NNF proposal reference lists. Keys are normalized proposal strings; values are
# the normalized journal_key used in the authority (remove_stopwords(normalize_journal_text(full_name))).
# Add more entries here when you see high-frequency unmatched strings in the diagnostics export.
KNOWN_ALIASES = {
    # General / multi-field
    'nejm': 'new england journal medicine',
    'new engl j med': 'new england journal medicine',
    'n engl j med': 'new england journal medicine',
    'engl. j. med': 'new england journal medicine',
    'bmj': 'british medical journal',
    'pnas': 'proceedings national academy sciences',
    'proc natl acad sci u s a': 'proceedings national academy sciences',
    'proc natl acad sci usa': 'proceedings national academy sciences',
    'natl acad sci u s a': 'proceedings national academy sciences',
    'proc natl acad sci': 'proceedings national academy sciences',
    'proc. natl. acad. sci. u. s. a': 'proceedings national academy sciences',
    'natl. acad. sci. u. s. a': 'proceedings national academy sciences',
    'natl. acad. sci': 'proceedings national academy sciences',
    'sci adv': 'science advances',
    # Clinical investigation
    'jci': 'journal clinical investigation',
    'j clin invest': 'journal clinical investigation',
    # Annals
    'ann intern med': 'annals internal medicine',
    'ann rheum dis': 'annals rheumatic diseases',
    'ann surg': 'annals surgery',
    'ann neurol': 'annals neurology',
    # Biochemistry / biophysics
    'biochim biophys acta': 'biochimica biophysica acta',
    'bba': 'biochimica biophysica acta',
    'free radic biol med': 'free radical biology medicine',
    # Frontiers journals
    'front immunol': 'frontiers immunology',
    'front microbiol': 'frontiers microbiology',
    'front physiol': 'frontiers physiology',
    'front neurosci': 'frontiers neuroscience',
    'front oncol': 'frontiers oncology',
    'front genet': 'frontiers genetics',
    'front cell dev biol': 'frontiers cell developmental biology',
    'front endocrinol': 'frontiers endocrinology',
    'front endocrinol lausanne': 'frontiers endocrinology',
    'front endocrinol (lausanne': 'frontiers endocrinology',
    'front plant sci': 'frontiers plant science',
    # Nature reviews
    'nat rev drug discov': 'nature reviews drug discovery',
    'nat rev cancer': 'nature reviews cancer',
    'nat rev immunol': 'nature reviews immunology',
    'nat rev genet': 'nature reviews genetics',
    'nat rev mol cell biol': 'nature reviews molecular cell biology',
    'nat rev neurosci': 'nature reviews neuroscience',
    'nat rev cardiol': 'nature reviews cardiology',
    'nat rev dis primers': 'nature reviews disease primers',
    'nat protoc': 'nature protocols',
    # Chemistry
    'j. am. chem. soc': 'journal american chemical society',
    'j am chem soc': 'journal american chemical society',
    'jacs': 'journal american chemical society',
    'angew chem int ed': 'angewandte chemie international edition',
    'angew. chem. int. ed': 'angewandte chemie international edition',
    'chem. int. ed': 'angewandte chemie international edition',
    'chem soc rev': 'chemical society reviews',
    'chem. soc. rev': 'chemical society reviews',
    'acs catal': 'acs catalysis',
    'acs synth biol': 'acs synthetic biology',
    'j med chem': 'journal medicinal chemistry',
    'metab eng': 'metabolic engineering',
    'j. chem. theory comput': 'journal chemical theory computation',
    'j chem theory comput': 'journal chemical theory computation',
    # Genetics / molecular biology
    'am j hum genet': 'american journal human genetics',
    'am. j. hum. genet': 'american journal human genetics',
    'hum mol genet': 'human molecular genetics',
    'nat struct mol biol': 'nature structural molecular biology',
    # Physiology (subspecialty abbreviations)
    'am j physiol endocrinol metab': 'american journal physiology endocrinology metabolism',
    'am j physiol renal physiol': 'american journal physiology renal physiology',
    'am j physiol cell physiol': 'american journal physiology cell physiology',
    'am j physiol heart circ physiol': 'american journal physiology heart circulatory physiology',
    # Cardiology / vascular
    'eur heart j': 'european heart journal',
    'j am heart assoc': 'journal american heart association',
    'arterioscler thromb vasc biol': 'arteriosclerosis thrombosis vascular biology',
    'j cereb blood flow metab': 'journal cerebral blood flow metabolism',
    # Nephrology / bone
    'j bone miner res': 'journal bone mineral research',
    'j am soc nephrol': 'journal american society nephrology',
    'jasn': 'journal american society nephrology',
    # Infectious disease / immunology
    'j virol': 'journal virology',
    'clin infect dis': 'clinical infectious diseases',
    'j infect dis': 'journal infectious diseases',
    'infect immun': 'infection immunity',
    'plos pathog': 'plos pathogens',
    # Endocrinology / reproduction
    'hum reprod': 'human reproduction',
    'diabet med': 'diabetic medicine',
    'endocr rev': 'endocrine reviews',
    # Obstetrics / gynecology
    'am j obstet gynecol': 'american journal obstetrics gynecology',
    # Internal medicine / surgery
    'j intern med': 'journal internal medicine',
    'j gen intern med': 'journal general internal medicine',
    'arch intern med': 'archives internal medicine',
    # Annual reviews
    'annu rev biochem': 'annual review biochemistry',
    # Botany / plant science
    'new phytol': 'new phytologist',
    'j exp bot': 'journal experimental botany',
    # Epidemiology / public health
    'scand j public health': 'scandinavian journal public health',
    # Sports medicine
    'med sci sports exerc': 'medicine science sports exercise',
    # Obesity
    'int j obes': 'international journal obesity',
    'int j obes (lond': 'international journal obesity',
    # Pathology
    'am j pathol': 'american journal pathology',
    # Microbiology
    'microb cell fact': 'microbial cell factories',
    # Neurobiology
    'neurobiol dis': 'neurobiology disease',
    # Development
    'dev cell': 'developmental cell',
    # Nursing
    'j adv nurs': 'journal advanced nursing',
    # Visualized experiments
    'j vis exp': 'journal visualized experiments',
    # Optics
    'opt. express': 'optics express',
    # Fertility
    'fertil steril': 'fertility sterility',
    # Scandinavian / niche journals
    'acta neurol scand': 'acta neurologica scandinavica',
    'acta oncol': 'acta oncologica',
    'dan med j': 'danish medical journal',
    'glob med genet': 'global medical genetics',
    # Materials / physics / engineering
    'jnucmat': 'journal nuclear materials',
    'nanoen': 'nano energy',
    'res in phys': 'results physics',
    'agrformet': 'agricultural forest meteorology',
    # Biotech
    'biotechnol bioeng': 'biotechnology bioengineering',
    # Omics / cell biology shorthand
    'molcel': 'mol cell',
    'celrep': 'cell reports',
    'cmet': 'cell metabolism',
    'omtn': 'molecular therapy nucleic acids',
    # Sensors / diagnostics
    'sens diagn': 'sensors diagnostics',
    # Other
    'j bone jt infect': 'journal bone joint infection',
    'jbji': 'journal bone joint infection',
    'lipids health dis': 'lipids health disease',
}

METADATA_COLS = [
    'journal_name', 'authority_source', 'openalex_source_id', 'issn_l', 'issn', 'abbreviated_title',
    'alternate_titles', 'publisher', 'homepage_url', 'works_count', 'cited_by_count', 'oa_2yr_mean_citedness',
    'oa_h_index', 'oa_i10_index', 'is_in_doaj', 'is_oa', 'is_core', 'host_organization_name',
    'mag_id', 'wikidata_id', 'fatcat_id'
]

def normalize_journal_text(text):
    text = str(text).lower().replace('&', ' and ')
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\b(the)\b', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def remove_stopwords(norm):
    return ' '.join(t for t in str(norm).split() if t not in STOPWORDS)

def abbreviation_variant(norm):
    toks = [t for t in str(norm).split() if t not in STOPWORDS]
    return ' '.join(ABBREV.get(t, t[:4] if len(t) > 6 else t) for t in toks)

def initials_variant(norm):
    toks = [t for t in str(norm).split() if t not in STOPWORDS]
    return ''.join(t[0] for t in toks if t)

def compact_variant(norm):
    return str(norm).replace(' ', '')

def journal_variants(name):
    norm = normalize_journal_text(name)
    base = remove_stopwords(norm)
    variants = {norm, base, abbreviation_variant(norm), abbreviation_variant(base), initials_variant(norm), compact_variant(norm), compact_variant(base)}
    return {v for v in variants if v and len(v) >= 2}

def ensure_cols(frame, cols):
    frame = frame.copy()
    for col in cols:
        if col not in frame.columns:
            frame[col] = pd.NA
    return frame[cols]

def parse_listish(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return value
    value = str(value).strip()
    if value.lower() in {'', 'null', 'none', 'nan'}:
        return []
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
        return [str(parsed).strip()]
    except Exception:
        return [value]

def read_openalex_sources(path):
    if not path.exists():
        return pd.DataFrame(columns=METADATA_COLS)
    out = pd.read_parquet(path)
    if 'source_type' in out.columns:
        out = out[out['source_type'].eq('journal')].copy()
    out['authority_source'] = 'openalex_sources'
    out = out[out['journal_name'].notna()].copy()
    return ensure_cols(out, METADATA_COLS)

def read_sciscinet_sources(path):
    if not path.exists():
        return pd.DataFrame(columns=METADATA_COLS)
    src = pd.read_parquet(path).rename(columns={
        'sourceid': 'openalex_source_id',
        'display_name': 'journal_name',
        'homepage_url': 'homepage_url',
        'is_oa': 'is_oa',
    })
    src['issn'] = src['issn'].map(parse_listish)
    src['issn_l'] = src['issn'].map(lambda xs: xs[0] if isinstance(xs, list) and len(xs) else pd.NA)
    src = src[src['journal_name'].notna()].copy()
    src['journal_name'] = src['journal_name'].astype(str).str.strip()
    src = src[src['journal_name'].ne('')]
    src = src[src['issn'].map(lambda xs: isinstance(xs, list) and len(xs) > 0)]
    non_journal_pat = re.compile(r'\b(ebooks?|conference|congress|symposium|proceedings|repository|workshop)\b', re.I)
    src = src[~src['journal_name'].str.contains(non_journal_pat, na=False)]
    src['authority_source'] = 'sciscinet_sources'
    return ensure_cols(src, METADATA_COLS)

def read_nnf_journals(path):
    if not path.exists():
        return pd.DataFrame(columns=METADATA_COLS)
    out = pd.read_parquet(path)
    name_col = 'journal_name' if 'journal_name' in out.columns else out.columns[0]
    out = out[[name_col]].rename(columns={name_col: 'journal_name'}).dropna()
    out['authority_source'] = 'proposal_output_journal_list'
    return ensure_cols(out, METADATA_COLS)

def read_wos_jcr(path):
    if not path.exists():
        return pd.DataFrame(columns=METADATA_COLS)
    raw = pd.read_csv(path, header=None)
    header_idx = raw.index[raw.iloc[:, 1].astype(str).str.lower().eq('full journal title')]
    if len(header_idx) == 0:
        raise ValueError('Could not find Full Journal Title row in wos_jcr.csv')
    out = raw.iloc[int(header_idx[0]) + 1:, [1]].rename(columns={1: 'journal_name'}).dropna()
    out['authority_source'] = 'wos_jcr'
    return ensure_cols(out, METADATA_COLS)


In [ ]:
authority_parts = [
    read_openalex_sources(OPENALEX_SOURCES_PATH),
    read_sciscinet_sources(SCISCINET_SOURCES_PATH),
    read_nnf_journals(NNF_JOURNALS_PATH),
    read_wos_jcr(WOS_JCR_PATH),
]
authority = pd.concat(authority_parts, ignore_index=True)
authority['journal_name'] = authority['journal_name'].astype(str).str.strip()
authority = authority[authority['journal_name'].ne('')].copy()
authority['journal_norm'] = authority['journal_name'].map(normalize_journal_text)
authority['journal_key'] = authority['journal_norm'].map(remove_stopwords)
priority = {'openalex_sources': 0, 'sciscinet_sources': 1, 'proposal_output_journal_list': 2, 'wos_jcr': 3}
authority['source_priority'] = authority['authority_source'].map(priority).fillna(9)
authority = authority.sort_values(['journal_key', 'source_priority', 'journal_name']).drop_duplicates('journal_key').reset_index(drop=True)
authority['journal_id'] = ['J%06d' % i for i in range(1, len(authority) + 1)]
authority.to_parquet(JOURNAL_AUTHORITY_PATH, index=False)

alias_rows = []
def add_alias(journal_id, alias, alias_type, authority_source):
    if pd.isna(alias) or not str(alias).strip():
        return
    alias = str(alias).strip()
    alias_norm = normalize_journal_text(alias)
    alias_rows.append({
        'journal_id': journal_id,
        'alias': alias,
        'alias_norm': alias_norm,
        'alias_key': remove_stopwords(alias_norm),
        'alias_type': alias_type,
        'authority_source': authority_source,
    })

for row in authority.itertuples(index=False):
    add_alias(row.journal_id, row.journal_name, 'canonical', row.authority_source)
    add_alias(row.journal_id, row.abbreviated_title, 'openalex_abbreviated_title', row.authority_source)
    for alt in parse_listish(row.alternate_titles):
        add_alias(row.journal_id, alt, 'openalex_alternate_title', row.authority_source)
    for variant in journal_variants(row.journal_name):
        add_alias(row.journal_id, variant, 'generated_variant', row.authority_source)

authority_aliases = pd.DataFrame(alias_rows).dropna(subset=['alias_key']).drop_duplicates(['journal_id', 'alias_key'])
authority_aliases.to_parquet(JOURNAL_AUTHORITY_ALIASES_PATH, index=False)

variant_to_ids = defaultdict(list)
for row in authority_aliases.itertuples(index=False):
    for variant in {row.alias_norm, row.alias_key, abbreviation_variant(row.alias_norm), compact_variant(row.alias_norm), compact_variant(row.alias_key)}:
        if variant and len(variant) >= 2:
            variant_to_ids[variant].append(row.journal_id)

for alias, canonical_key in KNOWN_ALIASES.items():
    hits = authority[authority['journal_key'].eq(canonical_key)]
    if len(hits):
        variant_to_ids[alias].append(hits.iloc[0]['journal_id'])

journal_id_to_name = dict(zip(authority['journal_id'], authority['journal_name']))
journal_id_to_key = dict(zip(authority['journal_id'], authority['journal_key']))
authority_by_first_token = defaultdict(list)
for row in authority.itertuples(index=False):
    toks = row.journal_key.split()
    if toks:
        authority_by_first_token[toks[0]].append(row)

print('authority journals:', len(authority))
print(authority['authority_source'].value_counts(dropna=False))
print('authority aliases:', len(authority_aliases))
authority[['journal_id', 'journal_name', 'authority_source', 'issn_l', 'abbreviated_title', 'oa_h_index', 'oa_i10_index', 'oa_2yr_mean_citedness', 'works_count', 'cited_by_count']].head()


# For final atypicality scoring, journal pairs must use IDs compatible with the
# background/null z-score table. For SciSciNet-v2, that should be OpenAlex sourceid.
journal_id_to_background_source_id = authority.set_index('journal_id')['openalex_source_id'].to_dict()
journal_id_to_authority_source = authority.set_index('journal_id')['authority_source'].to_dict()
print('authority journals with OpenAlex/SciSciNet source IDs:', int(authority['openalex_source_id'].notna().sum()))
print('authority journals without source IDs; useful only as review/helper aliases:', int(authority['openalex_source_id'].isna().sum()))


## 5. Match Parsed Journal Strings to Authority

This section matches parsed proposal journal strings to the authority list. It uses exact/variant matching first, then conservative fuzzy matching. Final scoreable IDs are OpenAlex/SciSciNet source IDs, not local unmatched clusters.


In [ ]:
try:
  from rapidfuzz import fuzz
  HAS_RAPIDFUZZ = True
except Exception:
  HAS_RAPIDFUZZ = False


JOURNAL_JUNK_TERMS = {
  '', 'na', 'n/a', 'none',
  'vol', 'volume', 'issue', 'no', 'number',
  'pp', 'page', 'pages',
  'chapter', 'book', 'press', 'publisher', 'isbn',
  'abstract', 'poster', 'presentation',
  'january', 'february', 'march', 'april', 'may', 'june',
  'july', 'august', 'september', 'october', 'november', 'december',
  'jan', 'feb', 'mar', 'apr', 'jun', 'jul', 'aug', 'sep', 'sept',
  'oct', 'nov', 'dec',
}


def is_bad_journal_candidate(candidate):
  if pd.isna(candidate) or not str(candidate).strip():
      return True

  raw = str(candidate).strip()
  norm = normalize_journal_text(raw)
  key = remove_stopwords(norm)

  if MONTH_RE.match(raw):
      return True

  if norm in JOURNAL_JUNK_TERMS or key in JOURNAL_JUNK_TERMS:
      return True

  if re.fullmatch(r'\d+', norm):
      return True

  if re.fullmatch(r'\d+\s*[-:]\s*\d+', norm):
      return True

  if re.match(r'^(vol|volume|issue|no|number|pp|pages?)\b', norm):
      return True

  alpha = re.sub(r'[^a-z]', '', norm)
  if len(alpha) < 3:
      return True

  return False


def choose_journal_id(ids):
  return sorted(set(ids), key=lambda x: len(journal_id_to_key.get(x, '')), reverse=True)[0]


def fuzzy_score(a, b):
  seq = SequenceMatcher(None, a, b).ratio()

  if HAS_RAPIDFUZZ:
      wratio = fuzz.WRatio(a, b) / 100
      token_set = fuzz.token_set_ratio(a, b) / 100
      return max(seq, wratio, token_set)

  return seq


def match_candidate_to_authority(candidate):
  if is_bad_journal_candidate(candidate):
      return None

  candidate = str(candidate).strip()
  norm = normalize_journal_text(candidate)
  key = remove_stopwords(norm)

  variants = [
      norm,
      key,
      abbreviation_variant(norm),
      abbreviation_variant(key),
      compact_variant(norm),
      compact_variant(key),
  ]

  variants = [v for v in dict.fromkeys(variants) if v and len(v) >= 2]

  # Tier 1: exact / generated-variant match using the existing authority index.
  for variant in variants:
      ids = variant_to_ids.get(variant, [])
      if ids:
          journal_id = choose_journal_id(ids)
          return {
              'match_method': 'candidate_exact_or_variant',
              'journal_id': journal_id,
              'journal_name': journal_id_to_name[journal_id],
              'journal_raw': candidate,
              'match_score': 1.0,
          }

  # Tier 2: conservative fuzzy match.
  toks = key.split()

  if len(key) >= 8 and toks:
      candidates = authority_by_first_token.get(toks[0], [])

      scored = []
      for row in candidates:
          score = fuzzy_score(key, row.journal_key)
          scored.append((score, row))

      scored = sorted(scored, key=lambda x: x[0], reverse=True)

      if scored:
          best_score, best_row = scored[0]
          second_score = scored[1][0] if len(scored) > 1 else 0.0
          score_gap = best_score - second_score

          # More conservative than the old >= .93 rule:
          # require a strong best score and separation from the runner-up.
          if best_score >= 0.93 and score_gap >= 0.04:
              return {
                  'match_method': 'candidate_fuzzy_authority',
                  'journal_id': best_row.journal_id,
                  'journal_name': best_row.journal_name,
                  'journal_raw': candidate,
                  'match_score': float(best_score),
              }

          # Allow near-exact fuzzy matches even if the gap is smaller.
          if best_score >= 0.98 and score_gap >= 0.01:
              return {
                  'match_method': 'candidate_fuzzy_authority',
                  'journal_id': best_row.journal_id,
                  'journal_name': best_row.journal_name,
                  'journal_raw': candidate,
                  'match_score': float(best_score),
              }

  return {
      'match_method': 'unmatched_candidate',
      'journal_id': pd.NA,
      'journal_name': pd.NA,
      'journal_raw': candidate,
      'match_score': np.nan,
  }


match_rows = []

for row in tqdm(parsed_refs.itertuples(index=False), total=len(parsed_refs), desc='Matching parsed journals'):
  rec = (
      match_candidate_to_authority(row.parsed_journal_raw)
      if row.is_paper_candidate and pd.notna(row.parsed_journal_raw)
      else None
  )

  if rec is None:
      rec = {
          'match_method': 'not_attempted_or_no_candidate',
          'journal_id': pd.NA,
          'journal_name': pd.NA,
          'journal_raw': pd.NA,
          'match_score': np.nan,
      }

  match_rows.append({
      ID_COL: getattr(row, '_0'),
      YEAR_COL: getattr(row, '_1'),
      'reference_position': row.reference_position,
      'reference_type': row.reference_type,
      'is_paper_candidate': row.is_paper_candidate,
      'has_doi': row.has_doi,
      'parser_used': row.parser_used,
      'parse_success': row.parse_success,
      'parsed_journal_raw': row.parsed_journal_raw,
      **rec,
  })

reference_matches = pd.DataFrame(match_rows)
reference_matches['matched_authority_journal'] = reference_matches['journal_id'].notna()

print(reference_matches['match_method'].value_counts(dropna=False))
print('authority-matched references:', int(reference_matches['matched_authority_journal'].sum()))

# Final scoring keeps only matches that resolve to an OpenAlex/SciSciNet source ID.
reference_matches['journal_background_source_id'] = reference_matches['journal_id'].map(journal_id_to_background_source_id)
reference_matches['journal_authority_source'] = reference_matches['journal_id'].map(journal_id_to_authority_source)
reference_matches['journal_final_id'] = reference_matches['journal_background_source_id']
reference_matches['journal_final_name'] = reference_matches['journal_name']
reference_matches['journal_match_tier'] = np.where(
    reference_matches['journal_background_source_id'].notna(),
    'openalex_or_sciscinet_source',
    np.where(reference_matches['matched_authority_journal'], 'authority_without_source_id', 'none')
)
reference_matches['has_final_journal'] = reference_matches['journal_final_id'].notna()

print('scoreable OpenAlex/SciSciNet-source references:', int(reference_matches['has_final_journal'].sum()))
print('authority matches lacking source IDs:', int(reference_matches['journal_match_tier'].eq('authority_without_source_id').sum()))


## 6. Optional Manual Corrections

If `manual_journal_matches.csv` exists, it should contain `journal_raw` and `journal_id`. The `journal_id` must exist in `journal_authority.parquet`. Manual corrections are useful for high-frequency unmatched strings, but they are scoreable only if the selected authority record has an OpenAlex/SciSciNet source ID.


In [ ]:
if MANUAL_JOURNAL_MATCH_CSV_PATH.exists():
    manual_df = pd.read_csv(MANUAL_JOURNAL_MATCH_CSV_PATH)
    required = {'journal_raw', 'journal_id'}
    missing = required - set(manual_df.columns)
    if missing:
        raise ValueError(f'Manual journal match CSV missing required columns: {missing}')
    manual_df = manual_df[['journal_raw', 'journal_id']].dropna().drop_duplicates()
    if manual_df['journal_raw'].duplicated().any():
        dupes = manual_df.loc[manual_df['journal_raw'].duplicated(keep=False), 'journal_raw'].drop_duplicates().head(20).tolist()
        raise ValueError(f'Manual CSV has duplicate journal_raw keys. Resolve first: {dupes}')
    bad_ids = sorted(set(manual_df['journal_id']) - set(authority['journal_id']))
    if bad_ids:
        raise ValueError(f'Manual CSV contains journal_id values not found in authority: {bad_ids[:20]}')

    manual_map = manual_df.set_index('journal_raw')['journal_id']
    manual_name_map = authority.set_index('journal_id')['journal_name'].to_dict()
    manual_source_map = authority.set_index('journal_id')['openalex_source_id'].to_dict()
    manual_authority_source_map = authority.set_index('journal_id')['authority_source'].to_dict()

    manual_mask = reference_matches['is_paper_candidate'] & reference_matches['journal_raw'].isin(manual_map.index)
    reference_matches.loc[manual_mask, 'journal_id'] = reference_matches.loc[manual_mask, 'journal_raw'].map(manual_map)
    reference_matches.loc[manual_mask, 'journal_name'] = reference_matches.loc[manual_mask, 'journal_id'].map(manual_name_map)
    reference_matches.loc[manual_mask, 'journal_background_source_id'] = reference_matches.loc[manual_mask, 'journal_id'].map(manual_source_map)
    reference_matches.loc[manual_mask, 'journal_authority_source'] = reference_matches.loc[manual_mask, 'journal_id'].map(manual_authority_source_map)
    reference_matches.loc[manual_mask, 'journal_final_id'] = reference_matches.loc[manual_mask, 'journal_background_source_id']
    reference_matches.loc[manual_mask, 'journal_final_name'] = reference_matches.loc[manual_mask, 'journal_name']
    reference_matches.loc[manual_mask, 'matched_authority_journal'] = True
    reference_matches.loc[manual_mask, 'journal_match_tier'] = np.where(
        reference_matches.loc[manual_mask, 'journal_background_source_id'].notna(),
        'manual_openalex_or_sciscinet_source',
        'manual_authority_without_source_id'
    )
    reference_matches.loc[manual_mask, 'match_method'] = 'manual_raw_to_authority'
    reference_matches.loc[manual_mask, 'match_score'] = 1.0
    reference_matches['has_final_journal'] = reference_matches['journal_final_id'].notna()
    print('manual mappings loaded:', len(manual_df))
    print('reference rows updated by manual mappings:', int(manual_mask.sum()))
else:
    print('No manual CSV found:', MANUAL_JOURNAL_MATCH_CSV_PATH)

reference_matches.to_parquet(REFERENCE_JOURNAL_MATCHES_PATH, index=False)


## 7. Diagnostics and Review Exports

This section clearly reports how much of the proposal citation material resolved to scoreable OpenAlex/SciSciNet sources. Unmatched and unscoreable journal strings are exported for manual review; they are not converted into local journal IDs.


In [ ]:
unmatched_strings = (
    reference_matches[reference_matches['is_paper_candidate'] & ~reference_matches['has_final_journal'] & reference_matches['journal_raw'].notna()]
    .groupby('journal_raw', dropna=False)
    .size()
    .reset_index(name='reference_count')
    .sort_values('reference_count', ascending=False)
)
unmatched_strings.to_parquet(UNMATCHED_REFERENCE_JOURNAL_STRINGS_PATH, index=False)
unmatched_strings.to_csv(CSV_DIR / 'unmatched_journal_strings_requiring_review.csv', index=False)

matched_dedup = reference_matches[reference_matches['has_final_journal']][[
    'journal_raw', 'journal_final_id', 'journal_final_name', 'journal_match_tier',
    'journal_id', 'journal_background_source_id', 'journal_authority_source',
    'journal_name', 'match_method', 'match_score', 'matched_authority_journal',
    'parser_used', 'parsed_journal_raw'
]].drop_duplicates().sort_values(['journal_match_tier', 'journal_final_name', 'journal_raw'])
matched_dedup.to_parquet(DEDUP_MATCHED_JOURNALS_PATH, index=False)
matched_dedup.to_csv(CSV_DIR / 'deduplicated_openalex_source_journal_matches.csv', index=False)

unmatched_dedup = reference_matches[reference_matches['is_paper_candidate'] & ~reference_matches['has_final_journal']][[
    'journal_raw', 'journal_name', 'journal_match_tier', 'match_method', 'parser_used', 'parsed_journal_raw'
]].drop_duplicates().sort_values(['journal_match_tier', 'match_method', 'journal_raw'])
unmatched_dedup.to_parquet(DEDUP_UNMATCHED_JOURNALS_PATH, index=False)
unmatched_dedup.to_csv(CSV_DIR / 'deduplicated_unmatched_or_unscoreable_journals.csv', index=False)

proposal_ref_summary = reference_matches.groupby([ID_COL, YEAR_COL], dropna=False).agg(
    parsed_reference_count=('reference_position', 'size'),
    paper_candidate_count=('is_paper_candidate', 'sum'),
    authority_journal_reference_count=('matched_authority_journal', 'sum'),
    scoreable_journal_reference_count=('has_final_journal', 'sum'),
    distinct_scoreable_journal_count=('journal_final_id', lambda x: x.dropna().astype(str).nunique()),
    no_scoreable_journal_paper_reference_count=('has_final_journal', lambda x: int((~x & reference_matches.loc[x.index, 'is_paper_candidate']).sum())),
).reset_index()
proposal_ref_summary['has_any_paper_reference_lacking_scoreable_journal'] = proposal_ref_summary['no_scoreable_journal_paper_reference_count'] > 0
proposal_ref_summary['has_two_or_more_distinct_scoreable_journals'] = proposal_ref_summary['distinct_scoreable_journal_count'] >= 2
proposal_ref_summary.to_parquet(PROPOSAL_MATCH_SUMMARY_PATH, index=False)
proposal_ref_summary.to_csv(CSV_DIR / 'proposal_reference_journal_match_summary.csv', index=False)

print('likely paper references:', int(reference_matches['is_paper_candidate'].sum()))
print('authority/manual matched paper references:', int((reference_matches['is_paper_candidate'] & reference_matches['matched_authority_journal']).sum()))
print('scoreable OpenAlex/SciSciNet-source paper references:', int((reference_matches['is_paper_candidate'] & reference_matches['has_final_journal']).sum()))
print('likely paper references without scoreable journal:', int((reference_matches['is_paper_candidate'] & ~reference_matches['has_final_journal']).sum()))
print('proposal units with at least one likely paper reference lacking scoreable journal:', int(proposal_ref_summary['has_any_paper_reference_lacking_scoreable_journal'].sum()))
print('proposal units with fewer than two distinct scoreable journals:', int((~proposal_ref_summary['has_two_or_more_distinct_scoreable_journals']).sum()))
print('matched journal export:', DEDUP_MATCHED_JOURNALS_PATH)
print('unmatched/unscoreable journal export:', DEDUP_UNMATCHED_JOURNALS_PATH)
unmatched_strings.head(25)


## 8. Build Proposal Source Pairs

For each proposal, take the distinct scoreable source IDs and form all unordered source pairs. These are the pairs that will later be looked up in the global background z-score table.


In [ ]:
def build_proposal_journal_pairs(reference_matches):
    # Only OpenAlex/SciSciNet source IDs are used. Unmatched/unscoreable journals are excluded.
    usable = reference_matches[reference_matches['has_final_journal']].copy()
    rows = []
    for (app_id, app_year), g in tqdm(usable.groupby([ID_COL, YEAR_COL], dropna=False), desc='Building proposal source pairs'):
        ids = sorted(g['journal_final_id'].dropna().astype(str).unique())
        total_pairs = len(ids) * (len(ids) - 1) // 2
        for j1, j2 in combinations(ids, 2):
            rows.append({
                ID_COL: app_id,
                YEAR_COL: app_year,
                'journal_1': j1,
                'journal_2': j2,
                'proposal_distinct_journals': len(ids),
                'proposal_total_pairs': total_pairs,
                'proposal_authority_pairs': total_pairs,
            })
    return pd.DataFrame(rows)

proposal_pairs = build_proposal_journal_pairs(reference_matches)
proposal_pairs.to_parquet(PROPOSAL_JOURNAL_PAIRS_PATH, index=False)
proposal_pairs.to_csv(CSV_DIR / 'proposal_openalex_source_pairs.csv', index=False)

print('proposal source pairs:', len(proposal_pairs))
print('proposal units with at least one source pair:', proposal_pairs[[ID_COL, YEAR_COL]].drop_duplicates().shape[0] if len(proposal_pairs) else 0)
proposal_pairs.head()


## 9. Score Proposal Atypicality Once Background Z-Scores Exist

Set `JOURNAL_PAIR_ZSCORE_PATH` after the SciSciNet-v2 background/null model has been built. Until then, this section correctly stops after building proposal source pairs.


In [ ]:
JOURNAL_PAIR_ZSCORE_PATH = None  # e.g. WORK_DIR / 'sciscinet_openalex_source_pair_zscores.parquet'
BACKGROUND_YEAR_OFFSET = -1

def read_zscore_table(path):
    path = Path(path)
    z = pd.read_parquet(path) if path.suffix.lower() == '.parquet' else pd.read_csv(path)
    z = z.rename(columns={'Year': 'year', 'JID1': 'journal_1', 'JID2': 'journal_2', 'Z_score': 'z_score'})
    needed = {'year', 'journal_1', 'journal_2', 'z_score'}
    missing = needed - set(z.columns)
    if missing:
        raise ValueError(f'z-score table missing columns: {missing}')
    z['journal_1'] = z['journal_1'].astype(str)
    z['journal_2'] = z['journal_2'].astype(str)
    swap = z['journal_1'] > z['journal_2']
    z.loc[swap, ['journal_1', 'journal_2']] = z.loc[swap, ['journal_2', 'journal_1']].to_numpy()
    return z[['year', 'journal_1', 'journal_2', 'z_score']].drop_duplicates()

def score_proposals(proposal_pairs, zscores):
    pairs = proposal_pairs.copy()
    pairs['background_year'] = pd.to_numeric(pairs[YEAR_COL], errors='coerce') + BACKGROUND_YEAR_OFFSET
    pairs['background_year'] = pairs['background_year'].astype('Int64')
    z = zscores.rename(columns={'year': 'background_year'})
    scored_pairs = pairs.merge(z, on=['background_year', 'journal_1', 'journal_2'], how='left')
    scored_pairs['pair_has_zscore'] = scored_pairs['z_score'].notna()
    summary = scored_pairs.groupby([ID_COL, YEAR_COL], dropna=False).agg(
        total_pairs=('z_score', 'size'),
        valid_pairs=('pair_has_zscore', 'sum'),
        atyp_z_median=('z_score', lambda x: np.nanmedian(x) if x.notna().any() else np.nan),
        atyp_z_10pct=('z_score', lambda x: np.nanquantile(x, 0.10) if x.notna().any() else np.nan),
    ).reset_index()
    summary['pair_coverage'] = summary['valid_pairs'] / summary['total_pairs'].replace({0: np.nan})
    summary['atypicality_score'] = -summary['atyp_z_10pct']
    return scored_pairs, summary

if JOURNAL_PAIR_ZSCORE_PATH:
    zscores = read_zscore_table(JOURNAL_PAIR_ZSCORE_PATH)
    scored_pairs, proposal_scores = score_proposals(proposal_pairs, zscores)
    scored_pairs.to_parquet(SCORED_PAIRS_PATH, index=False)
    proposal_scores.to_parquet(PROPOSAL_SCORES_PATH, index=False)
    proposal_scores.to_csv(CSV_DIR / 'proposal_atypicality_scores.csv', index=False)
    print('proposal scores saved:', PROPOSAL_SCORES_PATH)
    display(proposal_scores.head())
else:
    print('JOURNAL_PAIR_ZSCORE_PATH is None. Proposal journal pairs were built, but atypicality scores require a compatible background z-score table.')


## Remaining Work: Build the Background Z-Score Table

The proposal pipeline above stops at `proposal_pairs` unless `JOURNAL_PAIR_ZSCORE_PATH` is set. The missing background table is global/year-specific and should not be estimated from the proposal sample.

Target output file for scoring:

- `sciscinet_openalex_source_pair_zscores.parquet`
- Required columns: `year`, `journal_1`, `journal_2`, `z_score`
- `journal_1` and `journal_2` should be OpenAlex/SciSciNet source IDs, e.g. `S2764573992`.

Likely SciSciNet-v2 inputs:

- `sciscinet_paperrefs.parquet`: citing paper -> cited paper/reference edges.
- `sciscinet_papers.parquet`: paper year/type metadata.
- `sciscinet_papersources.parquet`: paper -> source ID; already inspected and present locally in the working directory.
- `sciscinet_sources.parquet`: source metadata; already used above.

Algorithm to adapt from the SciSciNet/MAG notebook:

1. Filter to journal/source papers with known publication year and source ID.
2. Build observed source-pair counts by citing year. For each citing paper, take the journals/sources of its cited papers and count all unordered source pairs.
3. Build null/reference lists by shuffling cited papers within `(citing_year, cited_year)` blocks, preserving citation age structure.
4. Repeat for multiple null epochs, e.g. 10 as in SciSciNet.
5. For each `(year, source_1, source_2)`, compute `z_score = (observed_count - mean(null_counts)) / std(null_counts)`.
6. Save the z-score table and set `JOURNAL_PAIR_ZSCORE_PATH` above.
